# 🛡️ SAFIR — Uctan Uca (End-to-End) Teknik Demonstrasyon

Bu defter, SAFIR'in **gercek calisan sisteminin teknik aynasidir**. Bir video verildiginde **production pipeline** (`src/main.py::SafirPipeline.run`) TEK cagriyla calisir; her asamanin **gercek** ara ciktisi bir gozlem (trace) kancasiyla yakalanip gosterilir.

> Notebook production mantigini kopyalamaz — yalnizca gercek kodu **import edip calistirir** ve yakalanan gercek ciktilari gorsellestirir. VLM olarak **Gemini** kullanilir.

### Gercek pipeline (koddan cikarilmis)
```
VIDEO
  ↓  AdaptiveFrameSampler.process_video        (src/sampler/adaptive_sampler.py)  -> kanit kareleri + motion_bbox
  ↓  AdaptiveFrameSampler.cluster_events                                          -> Olay Gruplari (zaman+IoU)
  ↓  RepresentativeFrameExtractor.extract      (src/sampler/context/...)          -> pre/peak/post kareler
  ↓  VLMClient/GeminiVLM.describe_events        (src/vlm/gemini_vlm.py)  → GEMINI  -> gozlem + EVENTS_JSON
  ↓  EventEngine.detect                         (src/event_analysis/event_engine.py) -> tipli olaylar
  ↓  TemporalReasoner.reason + RuleEngine.evaluate                                -> zamansal olaylar + ISG kurallari
  ↓  ContextBuilder.build                       (src/memory/context_builder.py)   -> ajan baglami (+RAG)
  ↓  SafirAgent.run                             (src/agent/langgraph_agent.py)    -> risk/ozet/aksiyon (JSON)
  ↓  EscalationPolicy.evaluate                  (src/decision/escalation.py)      -> otomatik eskalasyon
  ↓  SafirReport (+to_sartname_json)            (src/schemas/report.py)           -> FINAL REPORT
```

## 1) Environment & Configuration
Proje koku, konfigurasyon, cihaz ve model bilgileri. Eksik bagimlilik varsa **acik** hata verilir.

In [ ]:
import sys, os, platform
from pathlib import Path
_here = Path.cwd()
PROJECT_ROOT = _here if (_here / 'src').exists() else _here.parent
assert (PROJECT_ROOT / 'src').exists(), 'Proje koku (safir-ai/) bulunamadi.'
sys.path.insert(0, str(PROJECT_ROOT)); os.chdir(PROJECT_ROOT)

# ---- AYARLAR ----
USE_MOCK = False     # False: Gemini | True: offline (yalnizca gelistirici dogrulamasi)
USE_FAKE_RAG = True   # True: bge-m3 (~2GB) indirmez | False: gercek FAISS RAG

_missing = []
for _m in ['cv2', 'numpy', 'langgraph', 'langchain_openai', 'httpx', 'yaml', 'ipywidgets']:
    try: __import__(_m)
    except Exception as e: _missing.append(f'{_m} ({e})')
if _missing:
    raise ImportError('Eksik bagimlilik: ' + ', '.join(_missing) + '  ->  pip install -r requirements-gemini.txt')

from src.utils.config_loader import load_config
config = load_config()
if USE_MOCK:
    config = config.model_copy(update={'app': config.app.model_copy(update={'use_mock_vlm': True, 'use_mock_llm': True})})

print('Python      :', platform.python_version())
print('Device (cfg):', config.system.device)
print('VLM backend :', config.vlm.active_model, '->', config.vlm.models[config.vlm.active_model].model_name)
print('LLM backend :', config.llm.active_model, '->', config.llm.models[config.llm.active_model].model_name)
print('MOCK        :', USE_MOCK, '| FAKE_RAG:', USE_FAKE_RAG)
if not USE_MOCK and not os.environ.get('GEMINI_API_KEY'):
    print('\n[UYARI] GEMINI_API_KEY tanimli degil — Gemini cagrilari basarisiz olur. (Secret: ortam degiskeni)')

## 2) Input Video
Videoyu **kutudan sec** (surukle-birak) ya da yol yaz; hicbiri yoksa sentetik uretilir. Ardindan cozunurluk / FPS / sure / kare sayisi ve ornek kareler gorunur.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
uploader = widgets.FileUpload(accept='video/*', multiple=False, description='📹 Video sec')
path_box = widgets.Text(value='', placeholder='veya yol: data/ornek.mp4', description='Yol:')
display(widgets.VBox([uploader, path_box]))
print('Video secip ALTTAKI hucreyi calistir.')

In [ ]:
import tempfile, cv2, numpy as np
def _resolve_video():
    val = uploader.value
    if val:
        item = (list(val.values())[0] if isinstance(val, dict) else val[0])
        name = item.get('name') or (item.get('metadata', {}) or {}).get('name') or 'uploaded.mp4'
        Path('data').mkdir(exist_ok=True); p = f'data/{name}'
        Path(p).write_bytes(bytes(item['content'])); return p
    if path_box.value and Path(path_box.value).exists(): return path_box.value
    tmp = tempfile.mkdtemp(); p = str(Path(tmp) / 'synthetic.mp4')
    frames = [np.full((240, 320, 3), 30, np.uint8) for _ in range(75)]
    for i in range(25, 50): cv2.rectangle(frames[i], (40, 60), (260, 190), (200, 200, 200), -1)
    w = cv2.VideoWriter(p, cv2.VideoWriter_fourcc(*'mp4v'), 25.0, (320, 240))
    for f in frames: w.write(f)
    w.release(); print('(Sentetik video uretildi.)'); return p

VIDEO_PATH = _resolve_video()
cap = cv2.VideoCapture(VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS) or 0
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
DURATION = N_FRAMES / FPS if FPS else 0
print(f'Video       : {VIDEO_PATH}')
print(f'Cozunurluk  : {W}x{H}\nFPS         : {FPS:.1f}\nSure        : {DURATION:.1f} s\nKare sayisi : {N_FRAMES}')
row = []
for idx in [0, N_FRAMES // 2, max(0, N_FRAMES - 1)]:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx); ok, fr = cap.read()
    if ok:
        _ok, _b = cv2.imencode('.jpg', fr); row.append(widgets.Image(value=_b.tobytes(), format='jpeg', width=180))
cap.release()
print('Ornek kareler (bas / orta / son):'); display(widgets.HBox(row))

## 3) 🚀 Gercek Pipeline'i TEK Cagriyla Calistir (uctan uca)
Burada **production** `SafirPipeline.run()` **tek sefer** calisir. `trace` gozlem kancasi her asamanin **gercek** ara ciktisini yakalar (asagidaki bolumler bu yakalanan ciktilari gosterir). Asamalar gerceklestikce ✓ ile tik atar. **Gemini burada cagrilir.**

In [ ]:
if USE_FAKE_RAG:
    # bge-m3 (~2GB) indirmeden: RAG servisini hafif sahteyle degistir (yalnizca mevzuat metni)
    from dataclasses import dataclass
    import src.main as safir_main
    @dataclass
    class _Doc:
        text: str; score: float = 1.0
    class _FakeRAG:
        def seed_default_regulations(self): pass
        def query(self, q, top_k=None):
            return [_Doc('ISG Yonetmeligi Madde 24: KKD (baret/yelek) zorunludur.'),
                    _Doc('Operasyonel Kural OK-07: Forklift trafiginde yaya gecitleri acik tutulmalidir.')][:(top_k or 2)]
    safir_main.EmbeddingRAGService = lambda *a, **k: _FakeRAG()
else:
    import src.main as safir_main

USER_PROMPT = 'Sahnede riskli bir durum var mi degerlendir.'
captured = {}
_ORDER = ['sampler', 'clusters', 'vlm', 'events', 'agent_context', 'decision', 'escalation', 'report']
def trace(stage, payload):
    captured[stage] = payload
    print(f'  \u2713 {stage}')

print('Pipeline calisiyor (Gemini cagrilari dahil)...')
pipeline = safir_main.SafirPipeline(config)
report = pipeline.run(VIDEO_PATH, USER_PROMPT, trace=trace)
print('\nTAMAMLANDI. Yakalanan asamalar:', [s for s in _ORDER if s in captured])

In [ ]:
# Gorsel yardimcilari (yakalanan gercek ciktilari gostermek icin)
import base64
def _img(b, width=180): return widgets.Image(value=b, format='jpeg', width=width)
def _b64(data_uri): return base64.b64decode(data_uri.split(',', 1)[1])
def _draw_bbox(image_bytes, bbox):
    arr = cv2.imdecode(np.frombuffer(image_bytes, np.uint8), cv2.IMREAD_COLOR)
    if bbox:
        x0, y0, x1, y1 = bbox; cv2.rectangle(arr, (int(x0), int(y0)), (int(x1), int(y1)), (0, 0, 255), 2)
    _ok, buf = cv2.imencode('.jpg', arr); return buf.tobytes()

## 4) Cikti — Frame Sampling & Motion Region
**INPUT:** ham video · **PROCESSING:** `AdaptiveFrameSampler` (CPU; YOLO/ByteTrack yerine) · **OUTPUT:** kanit kareleri + hareket bolgesi (`motion_bbox`).

In [ ]:
st = captured['sampler']['stats']; ev = captured['sampler']['evidence_frames']
ratio = (100.0 * st.sampled_frames_evaluated / st.total_frames_scanned) if st.total_frames_scanned else 0
print(f'Original frames : {st.total_frames_scanned}')
print(f'Sampled frames  : {st.sampled_frames_evaluated}  (%{ratio:.1f})')
print(f'Evidence frames : {st.evidence_frame_count}  (elenen %{st.eliminated_ratio_pct} -> VLM tasarrufu)')
print('\nKanit kareleri + hareket bolgesi (kirmizi kutu = motion_bbox):')
display(widgets.HBox([_img(_draw_bbox(f.image_bytes, f.motion_bbox)) for f in ev]))
for f in ev: print(f'  [{f.timestamp_str}] change={f.change_score:.4f} motion_bbox={f.motion_bbox}')

## 5) Cikti — Clustering / Tracking & VLM'e Giden Kareler
**PROCESSING:** `cluster_events` (zaman + bbox IoU sureklilik) + `RepresentativeFrameExtractor` · **OUTPUT:** Olay Gruplari ve her grup icin **VLM'e gonderilen** pre/peak/post kareler.

In [ ]:
cl = captured['clusters']['clusters']
print(f'{len(ev)} kanit karesi -> {len(cl)} Olay Grubu\n')
for c in cl:
    print(f"=== Olay #{c.event_id} ({c.start_time:.1f}-{c.end_time:.1f}s) — VLM'e {len(c.representative_frames)} kare ===")
    for rf in c.representative_frames: print(f'   • {rf.label:<11} @ {rf.timestamp_str}')
    display(widgets.HBox([_img(_b64(rf.base64_image)) for rf in c.representative_frames]))

## 6) Cikti — VLM (Gemini) Girdi & Ham Yanit
**INPUT (Gemini'ye giden):** pre/peak/post kareler + prompt · **PROCESSING:** Gemini · **OUTPUT:** modelin **ham** yaniti + yapilandirilmis olaylar (`EVENTS_JSON`).

In [ ]:
from src.prompts import VLM_OBSERVER_SYSTEM_PROMPT
vr = captured['vlm']['vlm_response']
vcl = captured['vlm']['clusters']
print('=== GEMINI INPUT ===')
print('Kullanici istemi:', captured['vlm']['user_prompt'])
print('Sistem istemi (ilk 200 krk):', VLM_OBSERVER_SYSTEM_PROMPT[:200], '...')
print(f"Gonderilen kare sayisi: {sum(len(c.representative_frames) for c in vcl)}")
print('\n=== GEMINI RAW OUTPUT ===\n')
print(vr.description)
print('\n=== Parse edilmis yapilandirilmis olaylar (EVENTS_JSON) ===')
for e in vr.structured_events: print('  ', e)
if vr.description.startswith('[HATA]'):
    print('\n[!] Gemini cagrisi basarisiz (retry sonrasi). Pipeline dayanikliligi devrede: degraded devam etti.')

## 7) Cikti — Event / Temporal Analysis
**PROCESSING:** `EventEngine.detect` → `TemporalReasoner.reason` → `RuleEngine.evaluate` · **OUTPUT:** tipli olaylar, zamansal sureklilik, tetiklenen ISG kurallari.

In [ ]:
de = captured['events']
print('Tespit edilen olaylar (DetectedEvent):')
for d in de['detected_events']:
    print(f'   {{"event": "{d.event_type}", "time": {d.timestamp:.1f}, "confidence": {d.confidence:.2f}}}')
print('\nZamansal olaylar (TemporalEvent — sureklilik):')
for t in de['temporal_events']:
    print(f'   {t.event_type:<22} tekrar={t.occurrence_count} sure={t.duration:.1f}s')
print('\nTetiklenen ISG kurallari (RuleMatch):')
for r in de['rule_matches']:
    print(f'   [{r.rule_id}] ({r.severity}) {r.rule_description}')

## 8) Cikti — Decision / Risk Evaluation
**PROCESSING:** `SafirAgent.run` (LangGraph + araclar) → `EscalationPolicy.evaluate` · **OUTPUT:** risk seviyesi, ozet, aksiyonlar, otomatik eskalasyon.

In [ ]:
dec = captured['decision']['decision']
esc = captured['escalation']['escalation']
print('Risk Level :', dec.risk_level.upper(), f'(skor {dec.risk_score}/100)')
print('Ozet       :', dec.summary)
print('Aksiyonlar :')
for a in dec.actions: print('   -', a)
print('\nOtomatik eskalasyon:', esc.tier.value, '| alarm otomatik tetiklendi:', esc.auto_dispatched)
print('Gerekce            :', esc.reason)

## 9) Final Report
`SafirPipeline` ciktisi: **sartname-uyumlu JSON** + insan-okur ozet (ayni tek kosunun sonucu).

In [ ]:
import json
print('===== JSON (sartname uyumlu) =====')
print(json.dumps(report.to_sartname_json(), ensure_ascii=False, indent=2))
print('\n===== Insan-okur ozet =====')
print('Video Summary      :', report.summary or report.natural_language_summary)
print('Detected Events    :', report.detected_event_types)
print('Critical Moments   :', [f'{e.timestamp:.1f}s: {e.description[:48]}' for e in report.timeline])
print('Risk Level         :', report.risk_level.upper(), f'({report.risk_score}/100)')
print('Recommended Actions:')
for a in (report.actions or [report.recommended_action]): print('   -', a)

## 10) SAFIR END-TO-END TEST

In [ ]:
vr = captured['vlm']['vlm_response']
gemini_ok = ('vlm' in captured) and not vr.description.startswith('[HATA]')
checks = [
    ('Video Input',      Path(VIDEO_PATH).exists()),
    ('Video Processing', 'sampler' in captured),
    ('Frame Sampling',   captured.get('sampler', {}).get('stats').evidence_frame_count > 0 if 'sampler' in captured else False),
    ('Clustering/Track', len(captured.get('clusters', {}).get('clusters', [])) > 0),
    ('Gemini Inference', gemini_ok),
    ('Output Parsing',   'events' in captured),
    ('Event Analysis',   len(captured.get('events', {}).get('detected_events', [])) > 0),
    ('Risk/Decision',    'decision' in captured),
    ('Structured JSON',  bool(report.to_sartname_json())),
    ('Final Report',     report.event_id is not None),
]
print('=' * 42)
print('        SAFIR END-TO-END TEST')
print('=' * 42)
for name, ok in checks:
    mark = '✓' if ok else '✗'
    print(f'  {name:<22} {mark}')
print('=' * 42)
overall = all(ok for _, ok in checks)
print('STATUS:', 'PASS' if overall else 'BLOCKED')
if not overall:
    failed = [n for n, ok in checks if not ok][0]
    print('\nFailed stage :', failed)
    if failed == 'Gemini Inference':
        print('Error        :', vr.description)
        print('Root cause   : Gemini cagrisi basarisiz (kota/model/anahtar). Pipeline dayanikli -> degraded rapor uretti.')
        print('Cozum        : GEMINI_API_KEY + config.yaml model_name (kotasi olan model) kontrol et.')